In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd
from PIL import Image
from math import ceil
import random




In [2]:
base_path = "/kaggle/input/cassava-leaf-disease-classification/"
test_img_dir = os.path.join(base_path, "test_images/")
sample_submission_path = os.path.join(base_path, "sample_submission.csv")
sample_submission = pd.read_csv(sample_submission_path)




In [3]:
def load_image(path, size=(64, 64)):
    """Load an image, resize to `size`, and keep original 0‑255 pixel values."""
    img = Image.open(path).convert("RGB")
    img = img.resize(size, Image.BILINEAR)
    return np.asarray(img, dtype=np.float32)


def normalized_distance(a, b):
    """
    Compute Euclidean distance between L2‑normalized flattened vectors.
    This is equivalent to 2 · (1 − cosine_similarity) and works better
    for raw‑pixel centroids.
    """
    a_flat = a.ravel()
    b_flat = b.ravel()
    a_norm = np.linalg.norm(a_flat)
    b_norm = np.linalg.norm(b_flat)
    if a_norm == 0:
        a_norm = 1.0
    if b_norm == 0:
        b_norm = 1.0
    a_flat = a_flat / a_norm
    b_flat = b_flat / b_norm
    return np.linalg.norm(a_flat - b_flat)




In [4]:
class ImageDataGenerator:
    """Simple generator that yields batches of images (as numpy arrays)."""

    def __init__(self, df, img_dir, batch_size=32, img_size=(64, 64, 3), shuffle=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.batch_size = batch_size
        self.img_h, self.img_w, self.img_c = img_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[
            idx * self.batch_size : (idx + 1) * self.batch_size
        ]
        batch_x = np.empty(
            (len(batch_indices), self.img_h, self.img_w, self.img_c), dtype=np.float32
        )
        for i, row_idx in enumerate(batch_indices):
            img_name = self.df.loc[row_idx, "image_id"]
            img_path = os.path.join(self.img_dir, img_name)
            batch_x[i] = load_image(img_path, size=(self.img_h, self.img_w))
        return batch_x

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)




In [5]:
train_csv_path = os.path.join(base_path, "train.csv")
train_img_dir = os.path.join(base_path, "train_images/")
train_df = pd.read_csv(train_csv_path)

IMG_SIZE = (224, 224, 3)
NUM_CLASSES = 5

class_sums = {c: np.zeros(IMG_SIZE, dtype=np.float64) for c in range(NUM_CLASSES)}
class_counts = {c: 0 for c in range(NUM_CLASSES)}

norm_vec_len = IMG_SIZE[0] * IMG_SIZE[1] * IMG_SIZE[2]
class_norm_sums = {
    c: np.zeros(norm_vec_len, dtype=np.float64) for c in range(NUM_CLASSES)
}

for idx, row in train_df.iterrows():
    img_name = row["image_id"]
    label = int(row["label"])
    img_path = os.path.join(train_img_dir, img_name)
    img_arr = load_image(img_path, size=IMG_SIZE[:2])  # (224,224,3) with raw values
    class_sums[label] += img_arr
    img_flat = img_arr.ravel()
    norm = np.linalg.norm(img_flat)
    if norm > 0:
        img_norm = img_flat / norm
    else:
        img_norm = img_flat
    class_norm_sums[label] += img_norm
    class_counts[label] += 1
    if (idx + 1) % 2000 == 0:
        print(f"Processed {idx + 1} / {len(train_df)} training images")

class_means = {
    c: (
        (class_sums[c] / class_counts[c])
        if class_counts[c] > 0
        else np.zeros(IMG_SIZE, dtype=np.float32)
    )
    for c in range(NUM_CLASSES)
}

norm_class_means = {}
for c in range(NUM_CLASSES):
    if class_counts[c] > 0:
        mean_vec = class_norm_sums[c] / class_counts[c]
        vec_norm = np.linalg.norm(mean_vec)
        if vec_norm > 0:
            norm_class_means[c] = mean_vec / vec_norm
        else:
            norm_class_means[c] = mean_vec
    else:
        norm_class_means[c] = np.zeros(norm_vec_len, dtype=np.float32)

raw_flat_class_means = {c: class_means[c].ravel() for c in range(NUM_CLASSES)}

print("Finished computing and normalizing class centroids.")




Processed 2000 / 18721 training images
Processed 4000 / 18721 training images
Processed 6000 / 18721 training images
Processed 8000 / 18721 training images
Processed 10000 / 18721 training images
Processed 12000 / 18721 training images
Processed 14000 / 18721 training images
Processed 16000 / 18721 training images
Processed 18000 / 18721 training images
Finished computing and normalizing class centroids.


In [6]:
test_generator = ImageDataGenerator(
    sample_submission,
    img_dir=test_img_dir,
    batch_size=16,  # keep same batch size; memory is still manageable at 224×224
    img_size=IMG_SIZE,
    shuffle=False,
)

pred_labels = []

# Use only the normalized (cosine‑like) distance, which yields higher accuracy.
RAW_WEIGHT = 0.0
NORM_WEIGHT = 1.0

for batch_idx in range(len(test_generator)):
    batch_imgs = test_generator[batch_idx]  # shape (B,224,224,3)
    for img in batch_imgs:
        img_flat = img.ravel()
        norm = np.linalg.norm(img_flat)
        if norm > 0:
            img_norm_flat = img_flat / norm
        else:
            img_norm_flat = img_flat

        blended_distances = []
        for c in range(NUM_CLASSES):
            norm_dist = np.linalg.norm(img_norm_flat - norm_class_means[c])
            raw_dist = np.linalg.norm(img_flat - raw_flat_class_means[c])
            blended = RAW_WEIGHT * raw_dist + NORM_WEIGHT * norm_dist
            blended_distances.append(blended)

        pred_labels.append(int(np.argmin(blended_distances)))
    if (batch_idx + 1) % 10 == 0:
        print(f"Processed {batch_idx + 1} / {len(test_generator)} batches")

sample_submission["label"] = pred_labels
output_path = "submission.csv"
sample_submission.to_csv(output_path, index=False)
print(f"Submission written to {output_path}")

Processed 10 / 168 batches
Processed 20 / 168 batches
Processed 30 / 168 batches
Processed 40 / 168 batches
Processed 50 / 168 batches
Processed 60 / 168 batches
Processed 70 / 168 batches
Processed 80 / 168 batches
Processed 90 / 168 batches
Processed 100 / 168 batches
Processed 110 / 168 batches
Processed 120 / 168 batches
Processed 130 / 168 batches
Processed 140 / 168 batches
Processed 150 / 168 batches
Processed 160 / 168 batches
Submission written to submission.csv
